In [ ]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import re
from datetime import datetime
import dateutil.parser as dp
import json
import csv
import traceback
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed, ProcessPoolExecutor
from threading import Lock
from queue import Queue
import threading
import urllib.parse

class CaixinNewsCrawler:
    def __init__(self, save_path="/Users/sylvie/Documents/caixinnews拆分2"):
        print("正在初始化爬虫...")
        
        # 设置保存路径
        self.save_path = save_path
        if not os.path.exists(self.save_path):
            os.makedirs(self.save_path)
        
        # 关键词列表
        self.keywords = [
            '职场焦虑', '职场压力', '工作焦虑', 
            '工作倦怠', '工作疲惫', '职场倦怠','员工关怀',
            '职场内卷', '职场竞争', '职场PUA','职场霸凌',
            '职场心理咨询', '职场心理健康', '职场关怀'
        ]
        
        print(f"共 {len(self.keywords)} 个关键词:")
        for i, kw in enumerate(self.keywords, 1):
            print(f"  {i:2d}. {kw}")
        
        # 总汇总文件
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.summary_csv = os.path.join(self.save_path, f"all_keywords_summary_{timestamp}.csv")
        
        # 初始化汇总CSV
        self.init_summary_csv()
        
        # 关键词进度文件
        self.keyword_progress_file = os.path.join(self.save_path, "keyword_progress.json")
        
        # 多线程锁
        self.csv_lock = Lock()
        self.url_lock = Lock()
        
        # 初始化requests session
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Cache-Control': 'max-age=0',
        })
        
        print(f"保存路径: {self.save_path}")
        print(f"汇总文件将保存到: {self.summary_csv}")
    
    def init_summary_csv(self):
        """初始化汇总CSV文件，包含关键词列"""
        if not os.path.exists(self.summary_csv):
            with open(self.summary_csv, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                writer.writerow(['关键词', '标题', '发布时间', '渠道', '作者', '摘要','URL'])
            print(f"✓ 已创建汇总CSV文件: {self.summary_csv}")
        else:
            print(f"✓ 使用现有汇总CSV文件: {self.summary_csv}")
    
    def add_to_summary_csv(self, keyword, article_data):
        """将数据添加到汇总CSV文件"""
        with self.csv_lock:
            try:
                with open(self.summary_csv, 'a', newline='', encoding='utf-8-sig') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        keyword,
                        article_data['标题'],
                        article_data['发布时间'],
                        article_data['渠道'],
                        article_data['作者'],
                        article_data['摘要'],
                        article_data['URL']
                    ])
                return True
            except Exception as e:
                print(f"✗ 添加到汇总CSV失败: {e}")
                return False
    
    def load_keyword_progress(self):
        """加载关键词进度"""
        if os.path.exists(self.keyword_progress_file):
            try:
                with open(self.keyword_progress_file, 'r', encoding='utf-8') as f:
                    progress = json.load(f)
                print(f"✓ 已加载关键词进度: 已处理 {progress['completed_count']}/{len(self.keywords)} 个关键词")
                return progress
            except Exception as e:
                print(f"✗ 加载关键词进度失败: {e}")
        
        return {
            'completed_keywords': [],
            'current_keyword_index': 0,
            'completed_count': 0,
            'total_keywords': len(self.keywords)
        }
    
    def save_keyword_progress(self, completed_keywords, current_index):
        """保存关键词进度"""
        progress = {
            'completed_keywords': completed_keywords,
            'current_keyword_index': current_index,
            'completed_count': len(completed_keywords),
            'total_keywords': len(self.keywords),
            'last_update': datetime.now().isoformat(),
            'summary_csv': self.summary_csv
        }
        
        with open(self.keyword_progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress, f, ensure_ascii=False, indent=2)
        
        print(f"✓ 关键词进度已保存: {len(completed_keywords)}/{len(self.keywords)}")
    
    def init_csv_for_keyword(self, keyword):
        """为特定关键词初始化CSV文件（移除正文列，适配已删除的content爬取）"""
        # 清理关键词，用于文件名
        clean_keyword = re.sub(r'[\\/*?:"<>|]', "_", keyword)
        csv_file = os.path.join(self.save_path, f"keyword_{clean_keyword}.csv")
        
        if not os.path.exists(csv_file):
            with open(csv_file, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                # 移除“正文”列，适配已删除的content爬取逻辑
                writer.writerow(['标题', '发布时间','渠道','作者','摘要','URL'])
            print(f"✓ 已创建关键词CSV文件: {csv_file}")
        else:
            print(f"✓ 使用现有关键词CSV文件: {csv_file}")
        
        return csv_file
    
    def init_driver(self):
        """初始化浏览器驱动 - 优化版"""
        print("正在初始化浏览器...")
        
        edge_options = Options()
        edge_options.add_argument("--blink-settings=imagesEnabled=false")
        edge_options.add_argument("--disable-gpu")
        edge_options.add_argument("--no-sandbox")
        edge_options.add_argument("--disable-dev-shm-usage")
        edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        edge_options.add_experimental_option('useAutomationExtension', False)
        
        # 设置更真实的User-Agent
        edge_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0")
        
        driver_path = r"/Users/sylvie/Downloads/edgedriver_mac64 3/msedgedriver"
        
        try:
            service = Service(executable_path=driver_path)
            self.driver = webdriver.Edge(service=service, options=edge_options)
            
            # 设置更短的超时时间
            self.driver.set_page_load_timeout(30)
            self.driver.implicitly_wait(5)
            self.driver.maximize_window()
            
            print("✓ 浏览器初始化成功")
            return True
            
        except Exception as e:
            print(f"✗ 浏览器初始化失败: {e}")
            return False
    
    def load_progress(self, keyword):
        """加载爬取进度"""
        progress_file = os.path.join(self.save_path, f"progress_{keyword}.json")
        if os.path.exists(progress_file):
            try:
                with open(progress_file, 'r', encoding='utf-8') as f:
                    progress_data = json.load(f)
                
                self.crawled_urls = set(progress_data.get('crawled_urls', []))
                print(f"✓ 已加载进度，有 {len(self.crawled_urls)} 条已爬取记录")
                return True
            except Exception as e:
                print(f"✗ 加载进度失败: {e}")
        
        self.crawled_urls = set()
        return False
    
    def save_progress(self, keyword):
        """保存爬取进度"""
        progress_file = os.path.join(self.save_path, f"progress_{keyword}.json")
        progress_data = {
            'crawled_urls': list(self.crawled_urls),
            'last_crawl_time': datetime.now().isoformat(),
            'keyword': keyword
        }
        
        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress_data, f, ensure_ascii=False, indent=2)
        
        print(f"✓ 进度已保存到: {progress_file}")
    
    def select_time_filter(self):
        """选择时间筛选器为'一年内' - 优化等待时间"""
        try:
            print("执行时间筛选...")
            time.sleep(5)  # 适当减少初始等待
            
            time_filter = self.driver.find_element(By.CSS_SELECTOR, ".search_result_select .el-select")
            self.driver.execute_script("arguments[0].click();", time_filter)
            time.sleep(1.5)  # 减少等待
            
            time_filter.click()
            time.sleep(0.3)
            
            actions = self.driver.switch_to.active_element
            for i in range(4):
                actions.send_keys(Keys.ARROW_DOWN)
                time.sleep(0.2)  # 减少按键间隔
            
            actions.send_keys(Keys.ENTER)
            print("✓ 时间筛选完成")
            time.sleep(2)
            
            return True
            
        except Exception as e:
            print(f"选择时间筛选器失败: {e}")
            return False
    
    def is_in_2025(self, time_str):
        """检查时间是否在2025年 - 添加详细日志"""
        if not time_str or time_str.strip() == "":
            print(f"[日期检查] 时间字符串为空或无效: '{time_str}'")
            return False
        
        print(f"[日期检查] 原始字符串: '{time_str}'")
        
        try:
            # 处理中文日期格式
            cleaned_str = re.sub(r'[年月日]', '-', time_str)
            cleaned_str = re.sub(r'[^\d\-]', '', cleaned_str)
            cleaned_str = cleaned_str.strip('-')
            
            print(f"[日期检查] 清理后: '{cleaned_str}'")
            
            if len(cleaned_str) < 4:
                return False
            
            try:
                dt = dp.parse(cleaned_str).date()
                result = dt.year == 2025
                print(f"[日期检查] 解析为: {dt}, 是2025年: {result}")
                return result
            except Exception as parse_error:
                print(f"[日期检查] 解析失败: {parse_error}")
                years = re.findall(r'\d{4}', time_str)
                if years:
                    result = years[0] == "2025"
                    print(f"[日期检查] 提取年份: {years[0]}, 是2025年: {result}")
                    return result
                return False
        except Exception as e:
            print(f"[日期检查] 异常: {e}")
            return False
    
    def click_load_more_until_end(self, max_clicks=50):
        """点击'加载更多'按钮 - 优化等待"""
        print(f"开始点击'加载更多'按钮，最多点击{max_clicks}次...")
        
        load_count = 0
        no_new_content_count = 0
        
        while load_count < max_clicks:
            try:
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1.5)  # 减少滚动等待
                
                load_more_btn = WebDriverWait(self.driver, 8).until(  # 减少等待超时
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), '加载更多')]"))
                )
                
                if not load_more_btn.is_displayed():
                    break
                
                before_count = len(self.get_current_articles())
                
                print(f"点击第 {load_count + 1} 次'加载更多'...")
                self.driver.execute_script("arguments[0].click();", load_more_btn)
                
                time.sleep(2)  # 减少新内容加载等待
                
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                
                after_count = len(self.get_current_articles())
                new_articles = after_count - before_count
                
                if new_articles > 0:
                    print(f"✓ 加载了 {new_articles} 篇新文章")
                    load_count += 1
                    no_new_content_count = 0
                else:
                    no_new_content_count += 1
                    print(f"⚠ 没有新内容加载，尝试次数: {no_new_content_count}")
                    
                    if no_new_content_count >= 2:
                        print("连续2次没有新内容，停止加载")
                        break
                
                print(f"当前总文章数: {after_count}")
                
            except TimeoutException:
                print("未找到'加载更多'按钮，可能已加载所有内容")
                break
            except Exception as e:
                print(f"点击'加载更多'时出错: {e}")
                break
        
        total_articles = len(self.get_current_articles())
        print(f"✓ 加载完成: 共点击 {load_count} 次，总文章数: {total_articles}")
        return total_articles
    
    def get_current_articles(self):
        """获取当前页面所有文章元素"""
        try:
            selectors = [
                ".search_result_item",
                "li.search_result_item",
                ".sr_item",
                "[class*='search_result']",
            ]
            
            for selector in selectors:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    return elements
            
            return []
        except:
            return []
    
    def extract_search_page_info(self, soup):
    # 初始化变量（严格缩进，4个空格）
        publish_date = ""
        channel = ""
        title = ""
        summary = "无描述"  # 初始默认值

        try:
            # 1. 提取标题
            title_tag = soup.select_one("h3.sr_title a")
            if title_tag:
                title = title_tag.get_text(strip=True)

            # 2. 提取发布时间和渠道
            date_channel_tag = soup.find("p", class_="sr_date_channel")
            if date_channel_tag:
                full_text = date_channel_tag.get_text(strip=True)
                date_patterns = [
                    r'(\d{4}年\d{1,2}月\d{1,2}日)',
                    r'(\d{4}-\d{1,2}-\d{1,2})',
                    r'(\d{4}/\d{1,2}/\d{1,2})'
                ]
                for pattern in date_patterns:
                    match = re.search(pattern, full_text)
                    if match:
                        publish_date = match.group(1)
                        break
                channel_tag = date_channel_tag.find("a")
                if channel_tag:
                    channel = channel_tag.get_text(strip=True)

            # 3. 提取摘要（仅这一次给summary赋值，杜绝后续覆盖）
            desc_tag = soup.select_one("p.sr_desc")
            if desc_tag:
                summary_text = desc_tag.get_text(strip=False, separator='\n')
                summary_text = re.sub(r'\s+', ' ', summary_text).strip()
                if summary_text:
                    summary = summary_text  # 仅提取到有效文本时更新，后续不再主动修改

            # ====================== 过滤逻辑（带调试，核心强化） ======================
            # 辅助方法：统计纯汉字+阿拉伯数字（兼容所有场景）
            def count_chinese_and_digits(text):
                if not text or text.strip() == "":
                    return 0
                # 严格匹配中文汉字和阿拉伯数字，排除所有其他字符
                valid_chars = re.findall(r'[\u4e00-\u9fff0-9]', text)
                return len(valid_chars)
            
        # 辅助方法：查找第一个标点位置
            def find_first_punctuation(text):
                # 定义标点符号集合（包括全角和半角）
                punctuations = r'[，。！？；,.;!?;]'
                match = re.search(punctuations, text)
                if match:
                    return match.start(), match.group()
                return -1, None

            # 调试输出1：查看过滤前的原始数据
            print(f"【过滤调试】原始摘要：{summary}")
            print(f"【过滤调试】标题：{title}")

            # 步骤1：先判断是否为初始无效值
            if not summary or summary.strip() == "" or summary == "无描述":
                print(f"【过滤调试】摘要为默认值/空值，直接判定无效")
            else:
                # 步骤2：查找第一个标点位置
                first_punct_index, punct_char = find_first_punctuation(summary)
                
                if first_punct_index > 0:  # 找到了标点，并且不在开头位置
                    # 获取第一个标点前的部分
                    before_punct = summary[:first_punct_index]
                    
                    # 统计第一个标点前部分的纯汉字+数字数量
                    valid_char_num = count_chinese_and_digits(before_punct)
                    print(f"【过滤调试】第一个标点位置：{first_punct_index}，标点字符：'{punct_char}'")
                    print(f"【过滤调试】标点前部分：{before_punct}")
                    print(f"【过滤调试】标点前有效字符数（汉字+数字）：{valid_char_num}")
                    
                    # 步骤3：判断是否需要删除标点前部分
                    if valid_char_num <= 5:
                        # 删除第一个标点及其之前的内容，只保留后面的部分
                        new_summary = summary[first_punct_index + len(punct_char):].strip()
                        
                        # 如果处理后结果为空，或者只剩空白字符，则设为"无描述"
                        if not new_summary or new_summary.strip() == "":
                            summary = "无描述"
                            print(f"【过滤调试】删除标点前部分后为空，重置为：{summary}")
                        else:
                            summary = new_summary
                            print(f"【过滤调试】删除标点前部分，新摘要：{summary}")
                    else:
                        print(f"【过滤调试】标点前有效字符数大于5，保留原摘要")
                else:
                    print(f"【过滤调试】未找到标点或标点在开头，不做处理")
                    # 如果没有标点，保持原样，但进行有效性检查
                    valid_char_num = count_chinese_and_digits(summary)
                    if valid_char_num <= 5:
                        summary = "无描述"
                        print(f"【过滤调试】无标点但有效字符数小于等于5，重置为：{summary}")

            # 调试输出2：查看最终结果
            print(f"【过滤调试】最终摘要：{summary}\n")
            
        except Exception as e:
            # 异常场景：强制重置所有变量，确保不返回无效数据
            print(f"【提取异常】错误信息：{e}")
            publish_date = ""
            channel = ""
            title = ""
            summary = "无描述"  # 异常时强制置为无效值

        return title, publish_date, channel, summary

    
    def extract_all_article_links(self):
        """提取所有文章链接（包含搜索页摘要）"""
        print("提取所有文章链接...")
        time.sleep(2.5)  # 减少等待
        
        self.scroll_page()
        article_links = []
        self.article_info_dict = {}  # 初始化文章信息字典（存储搜索页数据）
        
        try:
            result_items = self.driver.find_elements(By.CSS_SELECTOR, "li.search_result_item")
            print(f"找到 {len(result_items)} 个搜索结果项")
            
            for item in result_items:
                try:
                    item_html = item.get_attribute("outerHTML")
                    item_soup = BeautifulSoup(item_html, "html.parser")
                    
                    # 提取标题链接
                    title_link = item_soup.select_one("h3.sr_title a")
                    if title_link:
                        href = title_link.get("href")
                        if href and 'caixin.com' in href:
                            clean_href = href.split('?')[0] if '?' in href else href
                            
                            # 提取搜索页所有信息（包含摘要）
                            search_title, publish_date, channel, summary = self.extract_search_page_info(item_soup)
                            
                            # 存入字典（key=URL，value=搜索页数据）
                            self.article_info_dict[clean_href] = {
                                'title': search_title,
                                'publish_date': publish_date,
                                'channel': channel,
                                'summary': summary  # 搜索页提取的摘要
                            }
                            
                            if clean_href not in article_links:
                                article_links.append(clean_href)
                    
                except Exception as e:
                    continue
                    
        except Exception as e:
            print(f"查找搜索结果项失败: {e}")
        
        print(f"✓ 共提取到 {len(article_links)} 个有效链接")
        return article_links
    
    def scroll_page(self, max_scrolls=10):
        """滚动页面以加载所有内容 - 优化版"""
        print("滚动页面...")
        
        last_height = self.driver.execute_script("return document.body.scrollHeight")
        scroll_count = 0
        
        while scroll_count < max_scrolls:
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.2)  # 减少滚动后等待
            
            new_height = self.driver.execute_script("return document.body.scrollHeight")
            
            if new_height == last_height:
                break
                
            last_height = new_height
            scroll_count += 1
        
        print(f"页面滚动完成，滚动次数: {scroll_count}")
    
    def extract_article_detail_requests(self, url):
        """使用requests获取文章详情（仅补充作者，摘要来自搜索页）"""
        try:
            # 从搜索页信息字典中获取基础数据
            title = "无标题"
            publish_date = ""
            channel = ""
            summary = "无描述"
            
            if hasattr(self, 'article_info_dict') and url in self.article_info_dict:
                info = self.article_info_dict[url]
                title = info.get('title', '无标题')
                publish_date = info.get('publish_date', '')
                channel = info.get('channel', '')
                summary = info.get('summary', '无描述')  # 直接使用搜索页摘要
            
            # 提取作者（仅补充作者信息，无正文提取）
            author = "未知作者"
            try:
                response = self.session.get(url, timeout=10)
                if response.status_code != 200:
                    print(f"[请求失败] 状态码: {response.status_code}")
                else:
                    soup = BeautifulSoup(response.text, "html.parser")
                    # 第一优先级：author_baidu
                    author_tag = soup.find("span", id="author_baidu")
                    if author_tag and author_tag.text:
                        author = author_tag.text.strip().replace("作者：", "")
                    # 第二优先级：top-author
                    if author == "未知作者":
                        top_author = soup.select_one('.top-author')
                        if top_author:
                            author_text = top_author.get_text(strip=True)
                            if author_text:
                                author_text = re.sub(r'^(文|刷新|编辑|记者|撰稿|摄影)[\s|｜\s]*', '', author_text)
                                if '|' in author_text or '｜' in author_text:
                                    parts = re.split(r'[|｜]', author_text)
                                    candidate = parts[-1].strip()
                                    author = re.sub(r'^(财新|周刊|杂志)\s*', '', candidate)
                                else:
                                    author = re.sub(r'^(财新|周刊|杂志)\s*', '', author_text)
                    # 第三优先级：meta标签
                    if author == "未知作者":
                        author_meta = soup.find('meta', {'name': 'author'})
                        if author_meta:
                            author = author_meta.get('content', '').strip()
                        if author == "":
                            author_meta = soup.find('meta', {'property': 'article:author'})
                            if author_meta:
                                author = author_meta.get('content', '').strip()
            except Exception as e:
                print(f"[提取作者失败] {e}")
            
            # 组装文章数据（无正文字段）
            article_data = {
                '标题': title,
                '发布时间': publish_date,
                '渠道': channel,
                '作者': author if author.strip() else "未知作者",
                '摘要': summary,  # 搜索页提取的摘要
                'URL': url
            }
            
            return article_data
            
        except Exception as e:
            print(f"✗ 解析文章 {url} 时出错: {e}")
            return None
    
    def crawl_articles_multithread(self, urls, keyword_csv, keyword, max_workers=5, max_articles=0):
        """多线程并发爬取文章（适配搜索页摘要，移除正文逻辑）"""
        if max_articles and len(urls) > max_articles:
            urls = urls[:max_articles]
            print(f"限制爬取前 {max_articles} 篇文章")
        
        print(f"\n开始多线程爬取 {len(urls)} 篇文章，线程数: {max_workers}")
        
        # 筛选未爬取的URL
        todo_urls = []
        with self.url_lock:
            for url in urls:
                if url not in self.crawled_urls:
                    todo_urls.append(url)
        
        if not todo_urls:
            print("所有URL已爬取，跳过")
            return 0
        
        print(f"需要爬取 {len(todo_urls)} 篇新文章")
        
        success_count = 0
        fail_count = 0
        batch_size = 10  # 每10条保存一次
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 提交任务
            future_to_url = {
                executor.submit(self.extract_article_detail_requests, url): url 
                for url in todo_urls
            }
            
            keyword_batch_data = []
            
            for i, future in enumerate(as_completed(future_to_url)):
                url = future_to_url[future]
                
                try:
                    article_data = future.result(timeout=30)
                    
                    if article_data:
                        # 检查是否为2025年的文章
                        is_2025 = self.is_in_2025(article_data['发布时间'])
                        
                        if is_2025:
                            # 关键词CSV数据（移除正文列，适配新表头）
                            keyword_row_data = [
                                article_data['标题'],
                                article_data['发布时间'],
                                article_data['渠道'],
                                article_data['作者'],
                                article_data['摘要'],
                                article_data['URL']
                            ]
                            
                            keyword_batch_data.append(keyword_row_data)
                            
                            # 添加到汇总CSV
                            self.add_to_summary_csv(keyword, article_data)
                            
                            with self.url_lock:
                                self.crawled_urls.add(url)
                            
                            success_count += 1
                            print(f"[{i+1}/{len(todo_urls)}] ✓ 成功: {url[:50]}...")
                        else:
                            fail_count += 1
                            print(f"[{i+1}/{len(todo_urls)}] 跳过非2025年文章")
                    else:
                        fail_count += 1
                        print(f"[{i+1}/{len(todo_urls)}] ✗ 解析失败")
                
                except Exception as e:
                    fail_count += 1
                    print(f"[{i+1}/{len(todo_urls)}] ✗ 爬取失败: {e}")
                
                # 批量保存
                if len(keyword_batch_data) >= batch_size:
                    self.save_batch_to_csv_threadsafe(keyword_batch_data, keyword_csv)
                    keyword_batch_data = []
                    print(f"✓ 已保存 {batch_size} 条数据到关键词CSV")
        
        # 保存剩余数据
        if keyword_batch_data:
            self.save_batch_to_csv_threadsafe(keyword_batch_data, keyword_csv)
        
        print(f"\n✓ 多线程爬取完成! 成功: {success_count}, 失败: {fail_count}")
        return success_count
    
    def save_batch_to_csv_threadsafe(self, batch_data, csv_file):
        """线程安全的批量保存到CSV"""
        with self.csv_lock:
            try:
                with open(csv_file, 'a', newline='', encoding='utf-8-sig') as f:
                    writer = csv.writer(f)
                    writer.writerows(batch_data)
                print(f"✓ 已保存 {len(batch_data)} 条数据到关键词CSV")
            except Exception as e:
                print(f"✗ 保存关键词CSV失败: {e}")
    
    def crawl_keyword(self, keyword, max_load_clicks=50, max_articles=1000):
        """爬取单个关键词（适配搜索页摘要）"""
        print(f"\n{'='*60}")
        print(f"开始处理关键词: {keyword}")
        print(f"{'='*60}")
        
        # 初始化关键词CSV文件
        keyword_csv = self.init_csv_for_keyword(keyword)
        
        # 加载进度
        self.load_progress(keyword)
        
        # 构建搜索URL
        search_url = f"https://search.caixin.com/newsearch/caixinsearch?keyword={keyword}&x=0&y=0"
        
        try:
            print(f"访问搜索页面: {search_url}")
            self.driver.get(search_url)
            print(f"✓ 已访问: {search_url}")
            print(f"页面标题: {self.driver.title}")
            time.sleep(3)
        except Exception as e:
            print(f"✗ 访问搜索页面失败: {e}")
            return 0
        
        print("\n执行时间筛选（一年内）...")
        self.select_time_filter()
        
        print("\n加载所有文章...")
        total_articles = self.click_load_more_until_end(max_clicks=max_load_clicks)
        
        print("\n提取文章链接...")
        all_links = self.extract_all_article_links()
        
        if not all_links:
            print("✗ 未找到任何文章链接")
            return 0
        
        print(f"✓ 共找到 {len(all_links)} 篇文章链接")
        
        print("\n多线程爬取文章内容...")
        success_count = self.crawl_articles_multithread(
            urls=all_links,
            keyword_csv=keyword_csv,
            keyword=keyword,
            max_workers=5,
            max_articles=max_articles
        )
        
        self.save_progress(keyword)
        
        print(f"\n✓ 关键词 '{keyword}' 处理完成: 成功 {success_count} 篇文章")
        print(f"✓ 数据保存到关键词CSV: {keyword_csv}")
        print(f"✓ 数据已添加到汇总CSV: {self.summary_csv}")
        
        return success_count
    
    def merge_all_keyword_csvs(self):
        """合并所有关键词CSV文件到汇总文件（适配新表头）"""
        print("\n开始合并所有关键词CSV文件...")
        
        try:
            # 查找所有关键词CSV文件
            keyword_files = []
            for keyword in self.keywords:
                clean_keyword = re.sub(r'[\\/*?:"<>|]', "_", keyword)
                csv_file = os.path.join(self.save_path, f"keyword_{clean_keyword}.csv")
                if os.path.exists(csv_file):
                    keyword_files.append((keyword, csv_file))
            
            print(f"找到 {len(keyword_files)} 个关键词CSV文件")
            
            # 备份原有汇总文件
            if os.path.exists(self.summary_csv) and os.path.getsize(self.summary_csv) > 50:
                backup_file = self.summary_csv.replace('.csv', f'_backup_{datetime.now().strftime("%H%M%S")}.csv')
                os.rename(self.summary_csv, backup_file)
                print(f"✓ 已备份原汇总文件: {backup_file}")
            
            # 重新创建汇总文件
            with open(self.summary_csv, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                writer.writerow(['关键词', '标题', '发布时间', '渠道', '作者', '摘要', 'URL'])
            
            total_records = 0
            
            for keyword, csv_file in keyword_files:
                try:
                    with open(csv_file, 'r', newline='', encoding='utf-8-sig') as f:
                        reader = csv.reader(f)
                        headers = next(reader)  # 跳表头
                        
                        records = []
                        for row in reader:
                            if len(row) >= 6:  # 适配新表头（标题、时间、渠道、作者、摘要、URL）
                                record = [keyword] + row
                                records.append(record)
                        
                    # 写入汇总文件
                    if records:
                        with open(self.summary_csv, 'a', newline='', encoding='utf-8-sig') as f:
                            writer = csv.writer(f)
                            writer.writerows(records)
                        
                        print(f"✓ 关键词 '{keyword}': 添加了 {len(records)} 条记录")
                        total_records += len(records)
                    
                except Exception as e:
                    print(f"✗ 处理关键词 '{keyword}' 的CSV文件失败: {e}")
            
            print(f"\n✓ 合并完成! 总共 {total_records} 条记录")
            print(f"✓ 汇总文件: {self.summary_csv}")
            
            return total_records
            
        except Exception as e:
            print(f"✗ 合并CSV文件失败: {e}")
            traceback.print_exc()
            return 0
    
    def run_all_keywords(self, max_load_clicks=50, max_articles_per_keyword=1000):
        """运行所有关键词的爬取"""
        try:
            print("=" * 60)
            print("财新网多关键词新闻爬虫启动（摘要来自搜索页）")
            print("=" * 60)
            
            # 加载关键词进度
            progress = self.load_keyword_progress()
            completed_keywords = progress['completed_keywords']
            current_index = progress['current_keyword_index']
            
            print(f"已处理关键词: {completed_keywords}")
            print(f"从第 {current_index + 1} 个关键词继续")
            
            # 初始化浏览器
            if not self.init_driver():
                return 0
            
            total_success = 0
            
            # 遍历关键词
            for i in range(current_index, len(self.keywords)):
                keyword = self.keywords[i]
                
                print(f"\n[进度: {i+1}/{len(self.keywords)}] 处理关键词: {keyword}")
                
                if keyword in completed_keywords:
                    print(f"✓ 关键词 '{keyword}' 已处理过，跳过")
                    continue
                
                # 爬取当前关键词
                success = self.crawl_keyword(
                    keyword=keyword,
                    max_load_clicks=max_load_clicks,
                    max_articles=max_articles_per_keyword
                )
                
                total_success += success
                
                # 更新进度
                completed_keywords.append(keyword)
                self.save_keyword_progress(completed_keywords, i+1)
                
                # 关键词间延迟
                if i < len(self.keywords) - 1:
                    next_keyword = self.keywords[i+1]
                    delay = random.uniform(5, 10)
                    print(f"\n等待 {delay:.1f} 秒后开始下一个关键词: {next_keyword}")
                    time.sleep(delay)
            
            print("\n" + "=" * 60)
            print("所有关键词爬取完成!")
            print(f"总共成功爬取: {total_success} 篇文章")
            
            # 合并CSV
            merged_count = self.merge_all_keyword_csvs()
            print(f"汇总文件包含: {merged_count} 条记录")
            print(f"汇总文件: {self.summary_csv}")
            
            print("=" * 60)
            
            # 清除进度文件
            if os.path.exists(self.keyword_progress_file):
                os.remove(self.keyword_progress_file)
                print("✓ 关键词进度文件已清除")
            
            return total_success
            
        except KeyboardInterrupt:
            print("\n用户中断，保存当前进度...")
            if 'completed_keywords' in locals() and 'i' in locals():
                self.save_keyword_progress(completed_keywords, i)
            return total_success if 'total_success' in locals() else 0
            
        except Exception as e:
            print(f"\n✗ 爬虫运行出错: {e}")
            traceback.print_exc()
            return 0
        finally:
            if hasattr(self, 'driver'):
                self.close_browser()
                self.session.close()
    
    def close_browser(self):
        """关闭浏览器"""
        try:
            self.driver.quit()
            print("✓ 浏览器已关闭")
        except:
            pass


def main():
    """主函数"""
    print("财新网多关键词新闻爬虫配置（摘要来自搜索页）:")
    print("1. 自动选择'一年内'时间筛选")
    print("2. 自动点击'加载更多'按钮加载所有文章")
    print("3. 摘要从搜索页标蓝区域（p.sr_desc）提取")
    print("4. 提取标题、发布时间、作者、摘要、URL、渠道")
    print("5. 只保存2025年的文章")
    print("6. 使用多线程+requests快速爬取")
    print("7. 每个关键词独立保存CSV文件（无正文列）")
    print("8. 所有数据汇总到一个总CSV文件")
    print("9. 数据保存到: /Users/sylvie/Documents/caixinnews拆分2")
    print()
    
    # 初始化爬虫
    crawler = CaixinNewsCrawler()
    print("关键词列表:")
    for i, kw in enumerate(crawler.keywords, 1):
        print(f"  {i:2d}. {kw}")
    print()
    
    # 用户确认

    confirm = input("是否开始爬取所有关键词？(y/n, 默认y): ").strip().lower()
    if confirm == 'n':
        print("程序已退出")
        return
    
    try:
        success_count = crawler.run_all_keywords(
            max_load_clicks=50,
            max_articles_per_keyword=1000
        )
        
        if success_count > 0:
            print(f"\n✓ 成功爬取了 {success_count} 篇文章！")
            print(f"✓ 每个关键词的数据独立保存到CSV文件")
            print(f"✓ 所有数据已汇总到: {crawler.summary_csv}")
        
    except Exception as e:
        print(f"\n✗ 程序出错: {e}")
        traceback.print_exc()


if __name__ == "__main__":
    print("财新网多关键词新闻爬虫启动中...")
    main()

财新网多关键词新闻爬虫启动中...
财新网多关键词新闻爬虫配置（摘要来自搜索页）:
1. 自动选择'一年内'时间筛选
2. 自动点击'加载更多'按钮加载所有文章
3. 摘要从搜索页标蓝区域（p.sr_desc）提取
4. 提取标题、发布时间、作者、摘要、URL、渠道
5. 只保存2025年的文章
6. 使用多线程+requests快速爬取
7. 每个关键词独立保存CSV文件（无正文列）
8. 所有数据汇总到一个总CSV文件
9. 数据保存到: /Users/sylvie/Documents/caixinnews拆分2

正在初始化爬虫...
共 14 个关键词:
   1. 职场焦虑
   2. 职场压力
   3. 工作焦虑
   4. 工作倦怠
   5. 工作疲惫
   6. 职场倦怠
   7. 员工关怀
   8. 职场内卷
   9. 职场竞争
  10. 职场PUA
  11. 职场霸凌
  12. 职场心理咨询
  13. 职场心理健康
  14. 职场关怀
✓ 已创建汇总CSV文件: /Users/sylvie/Documents/caixinnews拆分2/all_keywords_summary_20251227_230035.csv
保存路径: /Users/sylvie/Documents/caixinnews拆分2
汇总文件将保存到: /Users/sylvie/Documents/caixinnews拆分2/all_keywords_summary_20251227_230035.csv
关键词列表:
   1. 职场焦虑
   2. 职场压力
   3. 工作焦虑
   4. 工作倦怠
   5. 工作疲惫
   6. 职场倦怠
   7. 员工关怀
   8. 职场内卷
   9. 职场竞争
  10. 职场PUA
  11. 职场霸凌
  12. 职场心理咨询
  13. 职场心理健康
  14. 职场关怀

财新网多关键词新闻爬虫启动（摘要来自搜索页）
已处理关键词: []
从第 1 个关键词继续
正在初始化浏览器...
✓ 浏览器初始化成功

[进度: 1/14] 处理关键词: 职场焦虑

开始处理关键词: 职场焦虑
✓ 使用现有关键词CSV文件: /Users/sylvie/Documents/caixinnews拆分2/k